# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [10]:
# 1. Use the full path for uv
#!/home/susivakumar/.local/bin/uv venv .venv --seed --clear

# 2. Use the full path for the new python to install packages
# 1. First, install the base PyTorch for CUDA 12.1
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install --force-reinstall torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install flashinfer==0.6.8.post1 flashinfer-cubin==0.6.8.post1
# 2. Then, install vLLM separately (it will find the compatible version)
!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install \
    flashinfer-python flashinfer-cubin \
    --index-url https://flashinfer.ai/whl/cu121/
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install \
#  torch==2.9.1 \
#  torchvision==0.24.1 \
#  torchaudio==2.9.1 \
#  vllm==0.14.0
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install "outlines==0.0.46"
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install "outlines==0.0.34"
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install vllm==0.5.4
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install pyairports
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install vllm
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install --force-reinstall torch==2.4.0 vllm==0.5.4 --index-url https://download.pytorch.org/whl/cu121
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install ipywidgets
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#!pip uninstall -y torchcodec sentence-transformers
#!pip install "sentence-transformers==3.3.1"
# 3. Register the kernel
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m pip install ipykernel
#!/home/susivakumar/private/151BMathematicalReasoning/.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

Looking in indexes: https://flashinfer.ai/whl/cu121/


### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
#MAX_TOKENS  = 4096
MAX_TOKENS  = 4096

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning.\n\n "
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{final_answer}\n"
    "Brief justification (<= 2 sentences).\n\n "
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician reasoning.\n\n"
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{X} where X is the answer choice.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{X}\n"
    "Brief justification (<= 2 sentences).\n\n"
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_REFLECTION_FREE = (
    "Pick ONLY ONE method to reevaluate the answer to the question."
    "MANDATORY RULE:"
    "- Final answer MUST be: \\boxed{X} where X = correct answer."
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}."
    "- \\boxed{answer} MUST EXIST within the first 100 tokens"
    "CONSTRAINTS:"
    "- DO NOT evaluate using more than ONE METHOD"
)

SYSTEM_PROMPT_REFLECTION_MCQ = (
    "Pick ONLY ONE method to reevaluate the answer to the question."
    "MANDATORY RULE:\n"
    "- Final answer MUST be: \\boxed{X} where X = correct option."
    "- \\boxed{answer} MUST EXIST within the first 100 tokens"
    "CONSTRAINTS:\n"
    "- DO NOT evaluate using more than ONE METHOD"
)

SYSTEM_PROMPT_REFLECTION_MCQ = (
    "Pick ONLY ONE method to reevaluate the answer to the question."
    "MANDATORY RULE:\n"
    "- Final answer MUST be: \\boxed{X} where X = correct option."
    "- \\boxed{answer} MUST EXIST within the first 100 tokens"
    "CONSTRAINTS:\n"
    "- DO NOT evaluate using more than ONE METHOD"
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def build_reflection_prompt(question: str, options: Optional[list]) -> tuple[str, str, str]:
    """Return (system_prompt, user_prompt, reflection_prompt) for a question."""
    system_prompt, user_prompt = build_prompt(question, options)
    if options:
        return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_MCQ
    return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_FREE

# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



In [5]:
from collections import Counter
from vllm import SamplingParams

def extract_boxed_answer(text):
    """Extracts the content inside the last \boxed{}."""
    matches = re.findall(r'\\boxed\{(.+?)\}', text)
    return matches[-1].strip() if matches else None

def get_self_consistency_answer(llm, tokenizer, prompt, n_samples=5):
    """
    Runs self-consistency using vLLM's generate method.
    """
    # 1. Define Sampling Parameters for vLLM
    sampling_params = SamplingParams(
        n=n_samples,               # Number of samples to generate
        temperature=0.7,           # Variety for self-consistency
        top_p=0.95,
        max_tokens=MAX_TOKENS,
    )

    # 2. vLLM takes a list of strings (prompts)
    # This is much faster than the old loop-based generation
    outputs = llm.generate([prompt], sampling_params)
    
    responses = []
    
    # 3. Process the multiple outputs generated by vLLM
    for output in outputs[0].outputs:
        decoded_text = output.text
        ans = extract_boxed_answer(decoded_text)
        if ans:
            responses.append(ans)

    # 4. Perform Majority Vote
    if not responses:
        return "No answer found"

    vote_count = Counter(responses)
    # Returns the most common answer
    final_answer, count = vote_count.most_common(1)[0]
    return final_answer

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
import gc

# Delete model and tokenizer if they exist
try:
    del model
    del tokenizer
except:
    pass

# Clear Python garbage
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()

# Reset cached memory stats
torch.cuda.reset_peak_memory_stats()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
print("Memory cleared!")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Memory cleared!
Allocated: 0.00 GB
Cached: 0.00 GB


In [11]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side='left',
)

In [12]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side="left",
)

llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
)

INFO 05-03 05:42:51 [utils.py:263] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 05-03 05:42:51 [model.py:530] Resolved architecture: Qwen3ForCausalLM
WARNING 05-03 05:42:51 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 05-03 05:42:51 [model.py:1545] Using max model len 2048
INFO 05-03 05:42:51 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1571) INFO 05-03 05:43:04 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='a

(EngineCore_DP0 pid=1571) Process EngineCore_DP0:
(EngineCore_DP0 pid=1571) Traceback (most recent call last):
(EngineCore_DP0 pid=1571)   File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=1571)     self.run()
(EngineCore_DP0 pid=1571)   File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=1571)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=1571)   File "/home/susivakumar/private/151BMathematicalReasoning/.venv/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 940, in run_engine_core
(EngineCore_DP0 pid=1571)     raise e
(EngineCore_DP0 pid=1571)   File "/home/susivakumar/private/151BMathematicalReasoning/.venv/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 927, in run_engine_core
(EngineCore_DP0 pid=1571)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore_DP0 pid=1571)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [14]:
# Build prompts for first 5 entries
prompts = []
for item in data[:3]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate using self-consistency
print(f"Generating responses for {len(prompts)} questions...")
responses = []

for p in tqdm(prompts, desc="Processing Questions"):
    # generates N versions and picks the best
    final_answer = get_self_consistency_answer(llm, tokenizer, p, n_samples=5)
    responses.append(final_answer)

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i])
    # print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 3 questions...


Processing Questions: 100%|██████████| 3/3 [16:46<00:00, 335.64s/it]


── Response 0 (id=0) ──
3, 7

── Response 1 (id=1) ──
C

── Response 2 (id=2) ──
3, 7


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [10]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 1/1126 [00:00<01:44, 10.76it/s]

Scoring complete. 1 results.


## 8. Summary

Print accuracy broken down by question type.

In [11]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    0 /    0  (0.00%)
  Free-form  :    0 /    1  (0.00%)
  Overall    :    0 /    1  (0.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [12]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!